<a href="https://colab.research.google.com/github/mardyweb/atml-pa0/blob/main/notebooks/task1_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# to print nvidia driver's status table + to confirm that a gpu is attached to this session
!nvidia-smi

In [ ]:
from google.colab import userdata, drive
import os

USERNAME = "mardyweb"
REPO     = "atml-pa0"
TOKEN    = userdata.get('GITHUB_TOKEN')

# storing the url in an environment variable:
os.environ['GIT_URL'] = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

# clones the repo only if it is not already here (safe to re-run):
if not os.path.exists(f"/content/{REPO}"):
    !git clone $GIT_URL
%cd /content/$REPO

!git config user.email "maryamw17@outlook.com"
!git config user.name "Maryam"

# restoring the CIFAR-10 (170MB) and the ResNet-152 weights (230MB) that were cached to Drive
drive.mount('/content/drive')
!mkdir -p data /root/.cache/torch/hub/checkpoints
!cp -r /content/drive/MyDrive/atml_data/* data/ 2>/dev/null
!cp -r /content/drive/MyDrive/atml_cache/* /root/.cache/torch/hub/checkpoints/ 2>/dev/null
print("ready")

In [ ]:
from utils import set_seed, get_device, subset_loaders, save_results, save_fig

# should print cuda:
print(get_device())
# should print cifar-10-batches-py:
!ls data

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, time

# to ensure runs are reproducible:
set_seed(42)
device = get_device()

# subset_loaders loads CIFAR-10 (which has 60,000 images, 10 classes), resizes each to 224*224
# it then normalises the image with ImageNet's mean and std (ResNet's pretrained filters were trained on that size and pixel
# distribution) -> anything else would degarde the features badly (?)
# 5000 CIFAR images (500 per class) for training, 1000 for validation (100 per class)
# images are processed in batches of 64:
train_loader, val_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

model = torchvision.models.resnet152(weights="IMAGENET1K_V1")

# freeze the entire backbone (no gradients computed for these weights, weights are frozen)
for p in model.parameters():
    p.requires_grad = False

# model.fc is the final layer, currently 2048 -> 1000 (# of imagenet classes)
# we replace this with a 10-class CIFAR head
# requires_grad=True here:
model.fc = nn.Linear(model.fc.in_features, 10)

# all weights copied to GPU memory:
model = model.to(device)

# trainable params are 2048*10 (FC layer) + 10 biases = 20490/60M params
# (this is also why training from scratch is redundant)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

In [ ]:
# cross entropy is the multi-class classification loss
criterion = nn.CrossEntropyLoss()

# optimizer only accesses the params in the final layer (non-frozen):
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

def run_epoch(model, loader, train=False):
    """Does one complete pass over `loader`. Returns (average loss, accuracy)."""
    # model.eval() is used even when training. The backbone is frozen, so we do not want BatchNorm updating its running statistics on CIFAR data. nn.Linear
    # -> behaves identically in train/eval mode, so the head still learns normally.
    model.eval()
    total_loss, correct, n = 0.0, 0, 0

    # x: [64, 3, 224, 224] (64 images, 3 colour channels, 224x224 pixels)
    # y: [64] (the correct class index (0-9) for each image)

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            #gradients are cleared so that each batch isnt polluted by the previous batch:
            optimizer.zero_grad()
            # forward pass -> [64, 10] raw scores:
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            # applying the update to weights:
            optimizer.step()
        else:
            # no_grad() skips gradient bookkeeping entirely: faster, uses less
            # memory, and guarantees evaluation can't accidentally change
            # anything:
            with torch.no_grad():
                out = model(x)
                loss = criterion(out, y)

        # criterion returns average loss over the batch in loss, so we multiply by batch size before summing:
        total_loss += loss.item() * y.size(0)
        # argmax(1) picks highest scoring class per image
        correct    += (out.argmax(1) == y).sum().item()
        n          += y.size(0)

    return total_loss / n, correct / n

In [ ]:
# 5 passes over 5000 training images
EPOCHS = 5
hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    # updates weights:
    tl, ta = run_epoch(model, train_loader, train=True)
    # only measures:
    vl, va = run_epoch(model, val_loader,  train=False)

    hist["train_loss"].append(tl); hist["train_acc"].append(ta)
    hist["val_loss"].append(vl);   hist["val_acc"].append(va)

    print(f"epoch {ep}/{EPOCHS}  "
          f"train loss {tl:.4f} acc {ta:.4f} | "
          f"val loss {vl:.4f} acc {va:.4f}  ({time.time()-t0:.0f}s)")

#writes the metrics and exact config to results/task1_baseline.json
save_results("task1_baseline", {
    "config": {"model": "resnet152", "frozen_backbone": True, "epochs": EPOCHS,
               "lr": 0.001, "momentum": 0.9, "optimizer": "SGD",
               "n_train": 5000, "n_val": 1000, "batch_size": 64,
               "trainable_params": trainable, "total_params": total},
    "history": hist
})

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, EPOCHS + 1) #x-axis has the epoch number

# two plots side by side (loss curves, and accuracy)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ep, hist["train_loss"], label="Training Loss")
ax[0].plot(ep, hist["val_loss"],   label="Validation Loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[0].set_title("Loss over epochs"); ax[0].legend()

ax[1].plot(ep, hist["train_acc"], label="Training Accuracy")
ax[1].plot(ep, hist["val_acc"],   label="Validation Accuracy")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy")
ax[1].set_title("Accuracy over epochs"); ax[1].legend()

plt.tight_layout()
save_fig(fig, "task1_baseline_curves")
plt.show()

In [ ]:
# task 1.2:

import types

def forward_no_skip(self, x):
    """
    Replacement for a Bottleneck block's forward pass, with the residual removed.

    Normally this is what a ResNet block computes:
        out = F(x)        <- the three conv layers
        out = out + x     <- skip connection
        return relu(out)

    We drop that addition, so the block becomes three plain stacked convolutions.
    """
    out = self.conv1(x);   out = self.bn1(out);  out = self.relu(out)
    out = self.conv2(out); out = self.bn2(out);  out = self.relu(out)
    out = self.conv3(out); out = self.bn3(out)
    # we are removing the out += input from the original
    return self.relu(out)

# same seed as 1.1, so the new head starts from identical weights:
set_seed(42)

# building a second model, so we can compare with the 1.1 baseline model
model_ns = torchvision.models.resnet152(weights="IMAGENET1K_V1")

# freezing the backbone (identical to 1.1):
for p in model_ns.parameters():
    p.requires_grad = False

# final layer replaced with 10-class head:
model_ns.fc = nn.Linear(model_ns.fc.in_features, 10)

# ResNet-152's layer4 has 3 Bottleneck blocks: 0, 1, 2. we disable the skip connections in blocks 1 and 2
# not block 0 because it has a `downsample` branch that reshapes the tensor, so its skip path isn't a
# plain identity and removing it would change the block's output dimensions (?)

DISABLED = [1, 2]

for i in DISABLED:
    # getting the block object:
    blk = model_ns.layer4[i]
    # MethodType attaches the function to this block object only, so `self`
    # resolves to that block. every other block in the network keeps its residual connection.
    blk.forward = types.MethodType(forward_no_skip, blk)

model_ns = model_ns.to(device)
print(f"skip connections disabled in layer4 blocks {DISABLED}")

In [ ]:
# same seed means the same 5000 images as the baseline are used. Any difference in results is attributable to the missing skip connections,
# not different data:

set_seed(42)

train_loader, val_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

criterion = nn.CrossEntropyLoss()
# rebinding `optimizer` to the new model's head:
optimizer = optim.SGD(model_ns.fc.parameters(), lr=0.001, momentum=0.9)

hist_ns = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(model_ns, train_loader, train=True)
    vl, va = run_epoch(model_ns, val_loader,  train=False)
    hist_ns["train_loss"].append(tl); hist_ns["train_acc"].append(ta)
    hist_ns["val_loss"].append(vl);   hist_ns["val_acc"].append(va)
    print(f"epoch {ep}/{EPOCHS}  train loss {tl:.4f} acc {ta:.4f} | "
          f"val loss {vl:.4f} acc {va:.4f}  ({time.time()-t0:.0f}s)")

save_results("task1_noskip", {
    "config": {"disabled_blocks": DISABLED, "layer": "layer4",
               "epochs": EPOCHS, "lr": 0.001, "momentum": 0.9,
               "n_train": 5000, "n_val": 1000, "batch_size": 64},
    "history": hist_ns
})

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# left: training dynamics
ax[0].plot(ep, hist["train_loss"],    label="With Skip Connections")
ax[0].plot(ep, hist_ns["train_loss"], label="Without Skip Connections")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Training Loss")
ax[0].set_title("Training Loss Comparison"); ax[0].legend()

# right: generalisation
ax[1].plot(ep, hist["val_acc"],    label="With Skip Connections")
ax[1].plot(ep, hist_ns["val_acc"], label="Without Skip Connections")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation Accuracy")
ax[1].set_title("Validation Accuracy Comparison"); ax[1].legend()

plt.tight_layout()
save_fig(fig, "task1_skip_comparison")
plt.show()

print(f"final val acc — with skips: {hist['val_acc'][-1]:.4f} | "
      f"without: {hist_ns['val_acc'][-1]:.4f}")

In [ ]:
# task 1.3

import numpy as np

# a hook is a function PyTorch calls automatically whenever a particular layer produces output during a forward pass.
# hooks allow us to tap intermediate activation without modifying the model

features = {}

def make_hook(name):
    def hook(module, inp, out):
        # out from layer1 is [batch, 256, 56, 56] (a feature map per image)
        # Global average pooling collapses each 56x56 map to a single number by
        # averaging it, giving one value per channel: [batch, 256].
        # This is standard practice: it makes layers of different spatial sizes
        # directly comparable, and keeps t-SNE tractable.
        if out.dim() == 4:
            out = out.mean(dim=[2, 3])
        features[name] = out.detach().cpu()
    return hook

# register a hook on each of the four residual stages plus the final classifier
# deeper layers -> more abstract features
layers = {"layer1": model.layer1, "layer2": model.layer2,
          "layer3": model.layer3, "layer4": model.layer4, "fc": model.fc}

# attaches a hook to each of the five layers:
handles = [m.register_forward_hook(make_hook(n)) for n, m in layers.items()]

# collect features for the whole validation set (1000 images):
collected = {n: [] for n in layers}
labels = []
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        # hooks fire automatically here:
        model(x.to(device))
        for n in layers:
            collected[n].append(features[n])
        labels.append(y)

# removing hooks or they fire forever:
for h in handles:
    h.remove()

feats  = {n: torch.cat(v).numpy() for n, v in collected.items()} #what the network saw, this is a dict with 5 entries, one per layer
labels = torch.cat(labels).numpy() #what the image actually is

for n, f in feats.items():
    print(f"{n}: {f.shape}")   # (1000, 256) ... (1000, 2048), fc: (1000, 10)

In [ ]:
def separability(X, y, n_classes=10):
    """
    Ratio of between-class spread to within-class spread

    numerator   = mean distance between class centroids (how far apart classes sit)
    denominator = mean spread of points around their own centroid (how tight classes are)

    Higher = classes are further apart relative to their own scatter = more linearly separable
    ideally, it should increase with depth
    """

    centroids = np.stack([X[y == c].mean(0) for c in range(n_classes)])

    inter = [np.linalg.norm(centroids[i] - centroids[j])
             for i in range(n_classes) for j in range(n_classes) if i < j]

    intra = [np.linalg.norm(X[y == c] - centroids[c], axis=1).mean()
             for c in range(n_classes)]

    return float(np.mean(inter) / np.mean(intra))

scores = {n: separability(f, labels) for n, f in feats.items()}
for n, s in scores.items():
    print(f"{n}: {s:.4f}")

save_results("task1_separability", scores)

In [ ]:
!pip install -q umap-learn
from sklearn.manifold import TSNE
import umap, time

# both t-sne and umap reduce high-dimensional features to 2D for plotting
# t-SNE minimises KL divergence between pairwise-similarity distributions, preserves local neighbourhoods well
# but thedistances between clusters are not meaningful
# UMAP builds a k-nearest-neighbour graph and optimises a topological cross-entropy, preserves local structure and more global structure,
# and is usually much faster

embeddings, timings = {}, {}

for name in ["layer1", "layer2", "layer3", "layer4"]:
    # one layer's features (eg: [1000, 2048] for layer 4):
    X = feats[name]

    t0 = time.time()
    # squashing this to [1000,2] (2 numbers per image)
    embeddings[f"{name}_tsne"] = TSNE(n_components=2, random_state=42).fit_transform(X)
    timings[f"{name}_tsne"] = time.time() - t0

    t0 = time.time()
    embeddings[f"{name}_umap"] = umap.UMAP(n_components=2,
                                           random_state=42).fit_transform(X)
    timings[f"{name}_umap"] = time.time() - t0

    print(f"{name}: t-SNE {timings[f'{name}_tsne']:.1f}s | "
          f"UMAP {timings[f'{name}_umap']:.1f}s")

save_results("task1_reduction_times", timings)

In [ ]:
from utils import CIFAR10_CLASSES

# one 2x4 grid instead of eight separate figures:

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, name in enumerate(["layer1", "layer2", "layer3", "layer4"]):
    for row, method in enumerate(["tsne", "umap"]):
        ax = axes[row, col]
        emb = embeddings[f"{name}_{method}"]
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10", s=4)
        ax.set_title(f"{name} — {method.upper()} "
                     f"(sep={scores[name]:.3f})", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

handles_leg = [plt.Line2D([], [], marker='o', ls='', color=plt.cm.tab10(i/9),
                          label=CIFAR10_CLASSES[i]) for i in range(10)]
fig.legend(handles=handles_leg, loc='lower center', ncol=10, fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, "task1_feature_hierarchy")
plt.show()

In [ ]:
#task 1.4

def run_epoch_tl(model, loader, opt, train=False):
    """
    Version of run_epoch for transfer learning, where the backbone is training.

    Two differences from the 1.1/1.2 version:
      - model.train() during training, so BatchNorm updates its running
        mean/variance. In 1.1 the backbone was frozen and we deliberately
        blocked that; here the backbone is learning, so it must track the
        new data's statistics.
      - the optimizer is passed in rather than read from a global, so there's
        no chance of accidentally updating the wrong model.
    """
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            opt.zero_grad()
            out  = model(x)
            loss = criterion(out, y)
            loss.backward()
            opt.step()
        else:
            with torch.no_grad():
                out  = model(x)
                loss = criterion(out, y)

        total_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += y.size(0)

    return total_loss / n, correct / n


def build_model(init, scope):
    """
    init  : "pretrained" (ImageNet weights) or "random" (from scratch)
    scope : "last_block" (train layer4 + fc) or "full" (train everything)
    this gives the four experiments
    """
    set_seed(42)   # identical head init in all four experiments for fair comparison

    weights = "IMAGENET1K_V1" if init == "pretrained" else None
    m = torchvision.models.resnet152(weights=weights)

    if scope == "last_block":
        for p in m.parameters():
            p.requires_grad = False
        # unfreezing the deepest conv layer:
        for p in m.layer4.parameters():
            p.requires_grad = True
    # if scope is "full": everything stays trainable

    m.fc = nn.Linear(m.fc.in_features, 10)   # built after freezing means it's trainable
    m = m.to(device)

    # # We use two learning rates: 1) the head starts random, so it needs big steps (1e-3).
    # # for a pretrained backbone, large updates would destroy those features, so it gets 1e-4
    # # A random backbone has nothing worth preserving,
    # # so it gets the full 1e-3 (larger learning rate)
    # back_lr = 1e-4 if init == "pretrained" else 1e-3

    back_lr = 1e-3   # same as head, keeps init as the only varying factor

    # splitting trainable weights into 2 groups so that optimiser treats them differently (was more useful when i was using diff learning rates, but keeping it doesnt hurt):
    head_params = list(m.fc.parameters())
    back_params = [p for n, p in m.named_parameters()
                   if p.requires_grad and not n.startswith("fc")]

    groups = [{"params": head_params, "lr": 1e-3}]
    if back_params:
        groups.append({"params": back_params, "lr": back_lr})
    opt = optim.SGD(groups, momentum=0.9)

    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return m, opt, n_params

In [ ]:
EXPERIMENTS = [
    ("pretrained", "last_block"),
    ("pretrained", "full"),
    ("random",     "full"),
    ("random",     "last_block"),
]

results_tl = {}

for init, scope in EXPERIMENTS:
    key = f"{init}_{scope}"
    print(f"\n=== {key} ===")

    set_seed(42)
    tl_loader, vl_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

    m, opt, n_params = build_model(init, scope)
    print(f"trainable params: {n_params:,}")

    h = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    t_start = time.time()

    for ep in range(1, EPOCHS + 1):
        trl, tra = run_epoch_tl(m, tl_loader, opt, train=True)
        vll, vla = run_epoch_tl(m, vl_loader, opt, train=False)
        h["train_loss"].append(trl); h["train_acc"].append(tra)
        h["val_loss"].append(vll);   h["val_acc"].append(vla)
        print(f"  epoch {ep}: train {trl:.4f}/{tra:.4f} | val {vll:.4f}/{vla:.4f}")

    # for best compute/accuracy trade-off:
    results_tl[key] = {"init": init, "scope": scope, "history": h,
                       "trainable_params": n_params,
                       "train_time_s": time.time() - t_start}

    # freeing GPU memory before the next 60M-param model:
    del m, opt
    torch.cuda.empty_cache()

save_results("task1_transfer", results_tl)

In [ ]:
KEYS = [f"{i}_{s}" for i, s in EXPERIMENTS]
ep = range(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

for k in KEYS:
    h = results_tl[k]["history"]
    ax[0].plot(ep, h["val_acc"],  label=k)
    ax[1].plot(ep, h["val_loss"], label=k)

ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Validation Accuracy")
ax[0].set_title("Validation Accuracy by Strategy"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation Loss")
ax[1].set_title("Validation Loss by Strategy"); ax[1].legend(fontsize=8)

finals = [results_tl[k]["history"]["val_acc"][-1] for k in KEYS]
ax[2].bar(range(len(KEYS)), finals)
ax[2].set_xticks(range(len(KEYS)))
ax[2].set_xticklabels([k.replace("_", "\n") for k in KEYS], fontsize=8)
ax[2].set_ylabel("Final Validation Accuracy"); ax[2].set_title("Final Accuracy")

plt.tight_layout()
save_fig(fig, "task1_transfer_strategies")
plt.show()

print(f"\n{'experiment':<24} {'params':>12} {'time(s)':>9} {'final acc':>10}")
for k in KEYS:
    r = results_tl[k]
    print(f"{k:<24} {r['trainable_params']:>12,} "
          f"{r['train_time_s']:>9.0f} {r['history']['val_acc'][-1]:>10.4f}")

In [ ]:
# task 1.5

# The timings were already collected in task 1.3
# We show layer1 (low-level features) against layer4 (high-level)
fig, ax = plt.subplots(2, 2, figsize=(11, 10))

for row, name in enumerate(["layer1", "layer4"]):
    for col, method in enumerate(["tsne", "umap"]):
        a = ax[row, col]
        emb = embeddings[f"{name}_{method}"]     # (1000, 2) coordinates
        # c=labels colours each dot by its true class, so visible clumping
        # means the features separate the classes.
        a.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10", s=5)
        a.set_title(f"{name} — {method.upper()} "
                    f"({timings[f'{name}_{method}']:.1f}s)", fontsize=11)
        a.set_xticks([]); a.set_yticks([])   # axes are meaningless in both
                                             # methods — only structure matters

plt.tight_layout()
save_fig(fig, "task1_tsne_vs_umap")
plt.show()

print("timings (s):")
# for k, v in timings.items():
#     print(f"  {k}: {v:.2f}")

# Save the table as structured data for the report.
timing_table = {}
for name in ["layer1", "layer2", "layer3", "layer4"]:
    t = timings[f"{name}_tsne"]
    u = timings[f"{name}_umap"]
    timing_table[name] = {"tsne_s": round(t, 2),
                          "umap_s": round(u, 2),
                          "speedup": round(t/u, 2)}

# Mean over layers 2-4 only: layer1's UMAP time includes one-off Numba JIT
# compilation, which overstates UMAP's true cost.
ts = [timings[f"{n}_tsne"] for n in ["layer2","layer3","layer4"]]
us = [timings[f"{n}_umap"] for n in ["layer2","layer3","layer4"]]
timing_table["mean_layers_2_4"] = {"tsne_s": round(sum(ts)/3, 2),
                                   "umap_s": round(sum(us)/3, 2),
                                   "speedup": round((sum(ts)/3)/(sum(us)/3), 2)}

save_results("task1_tsne_umap_timings", timing_table)

In [ ]:
from sklearn.metrics import confusion_matrix
from utils import CIFAR10_CLASSES

# Using `model` from 1.1 (frozen backbone, ~45% accuracy).
m_best = model

preds, trues = [], []
m_best.eval()
with torch.no_grad():
    for x, y in val_loader:
        out = m_best(x.to(device))
        preds.append(out.argmax(1).cpu())   # argmax = the predicted class
        trues.append(y)                     # the correct class

preds = torch.cat(preds).numpy(); trues = torch.cat(trues).numpy()

# Row i, column j = how many images of class i were predicted as class j
# diagonal is correct predictions; everything off it is an error
cm = confusion_matrix(trues, preds)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")     # darker = more images
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45)
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")

# Print the count in each cell; white text on dark cells for readability.
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max()/2 else "black")

plt.colorbar(im); plt.tight_layout()
save_fig(fig, "task1_confusion_matrix")
plt.show()

# Rank the off-diagonal entries to find the worst class pairs.
# i != j excludes the diagonal (correct predictions).
off = [(cm[i, j], CIFAR10_CLASSES[i], CIFAR10_CLASSES[j])
       for i in range(10) for j in range(10) if i != j]
off.sort(reverse=True)
print("top confusions (true -> predicted):")
for n, t, p in off[:6]:
    print(f"  {t} -> {p}: {n}")

save_results("task1_confusion", {"matrix": cm.tolist(),
                                 "classes": CIFAR10_CLASSES})

In [ ]:
def extract_feats(net, loader):
    """
    Same hook-based feature extraction as 1.3, wrapped as a function so it can
    be applied to any ResNet. Returns (features per layer, labels).
    """
    feats_l, labs = {n: [] for n in ["layer1","layer2","layer3","layer4"]}, []
    store = {}

    def mk(name):
        def hook(mod, inp, out):
            # global average pooling: each feature map -> one number,
            # so layers of different spatial sizes stay comparable
            store[name] = out.mean(dim=[2,3]).detach().cpu()
        return hook

    # getattr(net, "layer1") is the same as net.layer1, but lets us loop
    hs = [getattr(net, n).register_forward_hook(mk(n)) for n in feats_l]

    net.eval()
    with torch.no_grad():
        for x, y in loader:
            # hooks fire during this call:
            net(x.to(device))
            for n in feats_l:
                feats_l[n].append(store[n])
            labs.append(y)
    # removes hooks when done:
    for h in hs:
        h.remove()

    return ({n: torch.cat(v).numpy() for n, v in feats_l.items()},
            torch.cat(labs).numpy())


set_seed(42)
# ResNet-18: 18 layers vs 152
r18 = torchvision.models.resnet18(weights="IMAGENET1K_V1").to(device)
f18, l18 = extract_feats(r18, val_loader)

sep18 = {n: separability(f, l18) for n, f in f18.items()}

print(f"{'layer':<8} {'ResNet-18':>10} {'ResNet-152':>12}")
for n in ["layer1","layer2","layer3","layer4"]:
    print(f"{n:<8} {sep18[n]:>10.4f} {scores[n]:>12.4f}")

save_results("task1_resnet18_comparison",
             {"resnet18": sep18,
              "resnet152": {k: scores[k] for k in sep18}})

In [ ]:
# committing + pushing to git:

!git config --global core.editor true
!git add .
!git commit -m "Task 1.5: Optional experiments"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL